In [0]:
select * from com_edp_prd.cmpa_insights_internal_schema.zip_to_territory_mapping;
--select * from  com_edp_prd.cmpa_insights_internal_schema.patient360;



WITH phvs AS (
  SELECT *
  FROM com_edp_prd.cmpa_insights_internal_schema.patient360
),

-- Long form: (patient_id, npi, role)
role_long AS (
  SELECT PATIENT_ID, first_dx_hcp_5yr       AS NPI, 'FIRST_DX'  AS role FROM phvs WHERE first_dx_hcp_5yr IS NOT NULL
  UNION ALL
  SELECT PATIENT_ID, first_tx_hcp_5yr       AS NPI, 'FIRST_TX'  AS role FROM phvs WHERE first_tx_hcp_5yr IS NOT NULL
  UNION ALL
  SELECT PATIENT_ID, npi, 'TOP5_3YR' AS role
  FROM phvs
  LATERAL VIEW explode(array(
      most_seen_hcp1_3yr_ranked,
      most_seen_hcp2_3yr_ranked,
      most_seen_hcp3_3yr_ranked,
      most_seen_hcp4_3yr_ranked,
      most_seen_hcp5_3yr_ranked
  )) e AS npi
  WHERE npi IS NOT NULL
),

-- Count distinct patients per (NPI, role)
role_counts AS (
  SELECT NPI, role, COUNT(DISTINCT PATIENT_ID) AS cnt
  FROM role_long
  GROUP BY NPI, role
),

-- Pivot roles to columns
counts_pivot AS (
  SELECT
    NPI,
    MAX(CASE WHEN role = 'FIRST_DX'  THEN cnt END) AS FIRST_DX_COUNT,
    MAX(CASE WHEN role = 'FIRST_TX'  THEN cnt END) AS FIRST_TX_COUNT,
    MAX(CASE WHEN role = 'TOP5_3YR'  THEN cnt END) AS MOST_SEEN_TOP5_COUNT
  FROM role_counts
  GROUP BY NPI

),

-- Count unique patients per NPI across all roles
unique_patient_counts AS (
  SELECT NPI, COUNT(DISTINCT PATIENT_ID) AS UNIQUE_PATIENT_TOUCHPOINTS
  FROM role_long
  GROUP BY NPI
)

SELECT
  c.NPI,
  CONCAT_WS(' ', p.FIRST_NAME, p.LAST_NAME) AS HCP_NAME,
  p.FIRST_NAME,
  p.LAST_NAME,
  p.PRIMARY_SPECIALTY,
  p.ORGANIZATION_NAME,
  p.SECONDARY_SPECIALTY,
  CASE 
    WHEN p.PRIMARY_SPECIALTY LIKE '%Genetic%' OR p.SECONDARY_SPECIALTY LIKE '%Genetic%' THEN 'Geneticist'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Pediatrics%' THEN 'Pediatrician'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Psychiatry & Neurology%' 
      OR p.SECONDARY_SPECIALTY LIKE '%Neurodevelopmental Disabilities%' 
      OR p.PRIMARY_SPECIALTY LIKE '%Neurological Surgery%' THEN 'Psychiatry & Neurology'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Nurse Practitioner%' OR p.PRIMARY_SPECIALTY LIKE '%Physician Assistant%' THEN 'NPPA'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Internal Medicine%' OR p.SECONDARY_SPECIALTY LIKE '%Internal Medicine%' THEN 'PCP'
    WHEN p.PRIMARY_SPECIALTY LIKE '%Family Medicine%' OR p.SECONDARY_SPECIALTY LIKE '%Family Medicine%' THEN 'PCP'
    ELSE 'Others'
  END AS SPECIALTY_CATEGORY,
  p.HCO_PRIMARY_NPI,
  COALESCE(c.FIRST_DX_COUNT, 0)        AS FIRST_DX_COUNT,
  COALESCE(c.FIRST_TX_COUNT, 0)        AS FIRST_TX_COUNT,
  COALESCE(c.MOST_SEEN_TOP5_COUNT, 0)  AS MOST_SEEN_TOP5_COUNT,
  COALESCE(upc.UNIQUE_PATIENT_TOUCHPOINTS, 0) AS UNIQUE_PATIENT_TOUCHPOINTS
FROM counts_pivot c
LEFT JOIN unique_patient_counts upc
  ON c.NPI = upc.NPI
LEFT JOIN com_edp_prd.com_raw.kom_providers p
  ON c.NPI = p.NPI
ORDER BY UNIQUE_PATIENT_TOUCHPOINTS DESC, FIRST_DX_COUNT DESC, FIRST_TX_COUNT DESC, MOST_SEEN_TOP5_COUNT DESC;


